In [ ]:
!pip install ultralytics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import csv
import os
import torch
import cv2
from PIL import Image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 41.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="key")
project = rf.workspace("xmltoyolo-8vfdh").project("pistols-unk3m")
dataset = project.version(1).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pistols-1 in yolov8:: 100%|██████████| 10168/10168 [00:02<00:00, 3726.06it/s]


In [ ]:
#the data yaml file for yolo
data_yaml = """
path: /content/Pistols-1

train: /content/Pistols-1/train/images
val: /content/Pistols-1/valid/images
test: /content/Pistols-1/test/images

nc: 1
names: ['Pistol']
"""

#save the yaml file

with open('data.yaml', 'w') as f:
  f.write(data_yaml)

print("data yaml file created")

data yaml file created


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import torch, gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print("GPU memory cleared.")

clear_gpu()


GPU memory cleared.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.yaml").load("yolov8s.pt")

results = model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,

    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0002,

    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.0,
    fliplr=0.5,
    flipud=0.0,
    erasing=0.1,


    amp=True,

    project='runs/detect',
    name='yolo',
    save_period=10,
    exist_ok=True,
    val=True)

Transferred 355/355 items from pretrained weights
Ultralytics 8.3.203 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.1, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.yaml, momentum=0.937, mosaic=0.5, multi_scale=False, name=yolo, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, pe

In [ ]:
from google.colab import files
files.download('/content/runs/detect/yolo/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>